# Local AI Document Assistant — RAG + Ollama

A complete Google Colab project. Upload a PDF, ask questions about it, and receive answers grounded in retrieved passages.

**Stack:** Ollama + Qwen 2.5 1.5B, Sentence Transformers, ChromaDB, PyPDF, Gradio.

> Run the cells from top to bottom. This runs in the **Colab cloud runtime**, not on your college lab PC. Select a T4 GPU in Runtime → Change runtime type if available. Keep the Colab tab/session alive while using the app.

## 1. Check the Colab runtime

In [ ]:
!nvidia-smi
import os, platform
print("Python:", platform.python_version())


/bin/bash: line 1: nvidia-smi: command not found
Python: 3.13.18


## 2. Install system dependencies and Ollama

This installs Ollama inside the temporary Colab runtime. It does not install anything on the college computer.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.10.0-2bui

## 3. Start Ollama and download the language model

The first model download can take a few minutes. If the server does not start, rerun this cell once.

In [ ]:
import subprocess, time, requests, os, signal

# Helper to check if Ollama API is already online
def is_ollama_ready():
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return r.ok
    except Exception:
        return False

# Start Ollama only if it isn't running yet
if not is_ollama_ready():
    print("Starting Ollama server...")
    subprocess.run("pkill -f 'ollama serve' || true", shell=True)
    server_log = open("/tmp/ollama.log", "w")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    # Wait for the local API
    for attempt in range(30):
        if is_ollama_ready():
            print("Ollama API is ready.")
            break
        time.sleep(2)
    else:
        print(open("/tmp/ollama.log").read())
        raise RuntimeError("Ollama server did not start. Check the log above.")
else:
    print("Ollama API is already running and ready.")

# Pull the language model
print("Pulling model qwen2.5:1.5b (this may take a couple of minutes)...")
subprocess.run(["ollama", "pull", "qwen2.5:1.5b"], check=True)

# Confirm available models
print("Available models:")
print(requests.get("http://127.0.0.1:11434/api/tags").json())

Ollama API is already running and ready.
Pulling model qwen2.5:1.5b (this may take a couple of minutes)...
Available models:
{'models': [{'name': 'qwen2.5:1.5b', 'model': 'qwen2.5:1.5b', 'modified_at': '2026-11-26T04:03:25.29724513Z', 'size': 986061892, 'digest': '65ec06548149b04c096a120e4a6da9d4017ea809c91734ea5631e89f96ddc57b', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '1.5B', 'quantization_level': 'Q4_K_M', 'context_length': 32768, 'embedding_length': 1536, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools']}]}


## 4. Install Python libraries

ChromaDB stores vectors; Sentence Transformers creates embeddings; PyPDF extracts PDF text; Gradio provides the web interface.

In [ ]:
!pip -q install -U chromadb sentence-transformers pypdf gradio requests


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


## 5. Build the RAG pipeline

What happens:
1. Extract text from the uploaded PDF.
2. Split it into overlapping chunks.
3. Convert chunks into embedding vectors.
4. Store the vectors and text in ChromaDB.
5. Retrieve the most relevant chunks for each question.
6. Send only those retrieved passages to Qwen through Ollama.

In [ ]:
import os, re, uuid, requests, chromadb, torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from concurrent.futures import ThreadPoolExecutor

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"
LLM_MODEL = "qwen2.5:1.5b"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Use the Colab GPU for embeddings when available
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", embedding_device)

embedding_model = SentenceTransformer(EMBED_MODEL_NAME, device=embedding_device)

# Persistent database survives app interactions during this runtime
client = chromadb.PersistentClient(path="/content/rag_chroma_db")
collection_name = "document_chunks"

# Start with a clean collection for this notebook session
try:
    client.delete_collection(collection_name)
except Exception:
    pass
collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

def extract_pdf_text(pdf_path):
    try:
        reader = PdfReader(pdf_path)
    except Exception as e:
        raise ValueError(f"Failed to parse PDF file: {str(e)}")

    pages = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text_cleaned = re.sub(r'\s+', ' ', text).strip()
        if text_cleaned:
            pages.append((page_num, text_cleaned))
    return pages

def split_into_chunks(text, chunk_size=900, overlap=150):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
        start = end - overlap
    return chunks

def ingest_pdf(pdf_path):
    pages = extract_pdf_text(pdf_path)
    if not pages:
        raise ValueError("No selectable text found. This may be a scanned or image-only PDF.")

    ids, documents, metadatas = [], [], []

    # Efficient concurrent preparation
    def process_single_page(page_data):
        page_num, page_text = page_data
        page_chunks = split_into_chunks(page_text)
        return page_num, page_chunks

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(process_single_page, pages))

    for page_num, page_chunks in results:
        for chunk_index, chunk in enumerate(page_chunks):
            ids.append(str(uuid.uuid4()))
            documents.append(chunk)
            metadatas.append({"page": int(page_num), "chunk": int(chunk_index)})

    if not documents:
        raise ValueError("The PDF did not contain enough text to create chunks.")

    # Generate embeddings in optimal batch size
    embeddings = embedding_model.encode(
        documents,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=False
    ).tolist()

    # Efficiently wipe previous index without re-allocating client resource
    existing = collection.get()
    if existing and existing.get("ids"):
        collection.delete(ids=existing["ids"])

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )
    return len(pages), len(documents)

def retrieve_context(question, top_k=4):
    if collection.count() == 0:
        raise ValueError("Please upload and index a PDF document first.")
    query_embedding = embedding_model.encode(
        [question], normalize_embeddings=True
    ).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    passages = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        passages.append({
            "text": doc,
            "page": meta["page"],
            "distance": distance
        })
    return passages

def ask_rag(question):
    if not question or not question.strip():
        return "Please enter a valid question.", ""

    passages = retrieve_context(question)
    context = "\n\n".join(
        f"[Source: page {p['page']}]\n{p['text']}" for p in passages
    )

    system_prompt = (
        "You are a careful document question-answering assistant. "
        "Answer using only the supplied context. If the answer is not in the context, "
        "say you could not find it in the uploaded document. Do not invent facts. "
        "Cite page numbers when making claims."
    )
    user_prompt = f"DOCUMENT CONTEXT:\n{context}\n\nQUESTION: {question}"

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": LLM_MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "stream": False,
            "options": {"temperature": 0.15}
        },
        timeout=180
    )
    response.raise_for_status()
    answer = response.json()["message"]["content"]

    source_lines = []
    for i, p in enumerate(passages, start=1):
        source_lines.append(
            f"{i}. Page {p['page']} (retrieval distance: {p['distance']:.3f})"
        )
    return answer, "\n".join(source_lines)

print("Upgraded parallel RAG pipeline is ready.")

Embedding device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Upgraded parallel RAG pipeline is ready.


## 6. Create the document-chat interface

Upload a PDF, click **Index PDF**, then ask questions. The answer and retrieved source pages appear below.

In [ ]:
import gradio as gr

def index_uploaded_file(file_path):
    if file_path is None:
        return "Please upload a PDF first."
    try:
        page_count, chunk_count = ingest_pdf(file_path)
        return f"Indexed successfully: {page_count} pages converted into {chunk_count} searchable chunks."
    except Exception as e:
        return f"Indexing failed: {type(e).__name__}: {e}"

def answer_in_ui(question):
    try:
        answer, sources = ask_rag(question)
        return answer, sources
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}", ""

with gr.Blocks(title="Local AI Document Assistant") as demo:
    gr.Markdown(
        "# 📚 Local AI Document Assistant\n"
        "Upload a PDF and ask questions. Answers are generated by Qwen through Ollama "
        "using relevant passages retrieved from your document."
    )
    with gr.Row():
        pdf_file = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")
    index_button = gr.Button("Index PDF", variant="primary")
    index_status = gr.Textbox(label="Indexing status", interactive=False)
    index_button.click(index_uploaded_file, inputs=pdf_file, outputs=index_status)

    question = gr.Textbox(
        label="Ask a question",
        placeholder="Example: Explain the main concept discussed in this document..."
    )
    ask_button = gr.Button("Ask", variant="primary")
    answer = gr.Markdown(label="Answer")
    sources = gr.Textbox(label="Retrieved source pages", lines=5, interactive=False)
    ask_button.click(answer_in_ui, inputs=question, outputs=[answer, sources])
    question.submit(answer_in_ui, inputs=question, outputs=[answer, sources])

    gr.Markdown(
        "**Note:** This is a prototype. Scanned PDFs need OCR, and answers depend on "
        "the quality of the extracted text and retrieved passages."
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3c3fee6b497554b46c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3c3fee6b497554b46c.gradio.live
